# Preprocessing, Feature Engineering & Hyperparameter Tuning

## Concepts Learned

1. Numeric Processesing
    * Normalization - Convert Numeric Values to be between 0 and 1
    * Standardization - Conver NumeriC Values to Z-scores
2. Catogerical Processesing
    * One hot encoding - Get Dummies
    * Oridianl Encoding - Covert each unique value in a Cateogrical column into a number. 
3. Feature Selection
    * L1 - Embedded Feature Selection. Makes it so that features with no usefullness automatically have a weight of 0
4. Hyperparamter
    * GridSearchCV - Tests every combination of given values for each parameter and finds the best one.
    * RandomSearchCV - Requires inteverals for each paramater and randomliy guess each paramter. It can easily find the globabl maximum but is computitianal expensive.
5. Data Leakage
    * When testing and training dataset are not independent
    * Cases:
        * When you use tesing dataset to train the model
        * When you impute before spliting. ALWAYS IMPUTE, ENCODE, SCALE, NORMALIZE AFTER SPLITTING

In [60]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.linear_model import Lasso
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor

import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("car_data.csv")
data.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


In [16]:
data.isna().sum()

year             0
selling_price    0
km_driven        0
fuel             0
seller_type      0
transmission     0
owner            0
dtype: int64

In [4]:
data.drop(columns=['name'], inplace=True)

In [5]:
X = data.drop(columns=['selling_price'])
y = data['selling_price']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, shuffle=True)

In [10]:
data['transmission'].value_counts()

transmission
Manual       3892
Automatic     448
Name: count, dtype: int64

In [24]:
data['fuel'].value_counts()

fuel
Diesel      2153
Petrol      2123
CNG           40
LPG           23
Electric       1
Name: count, dtype: int64

In [33]:
scaler = StandardScaler()
onehot = OneHotEncoder(handle_unknown='ignore')
ordinial = OrdinalEncoder()
mms = MinMaxScaler()

In [43]:


col_transformation = ColumnTransformer([
    ("OneHotEncoding", onehot, ['fuel', 'seller_type', 'owner']),
    ("OrdinalEncoder", ordinial, ['transmission']),
    ("Scaling", scaler, ['km_driven', 'year'])
])

In [62]:
pipeline = Pipeline([
    ("Preprocessing", col_transformation),
    ("feature_creation", PolynomialFeatures(degree=2, interaction_only=False)),
    ("model", Lasso())
])


TTR = TransformedTargetRegressor(
    regressor=pipeline,
    transformer=StandardScaler()
)

In [64]:
param_grid = {
    "regressor__model__alpha" : [0.1,0.01,0.001,1]
}

gs = GridSearchCV(
    estimator=TTR,
    cv = 5,
    param_grid=param_grid,
    scoring="r2",
    refit='r2'
)

In [65]:
gs.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",TransformedTa...ndardScaler())
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__model__alpha': [0.1, 0.01, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'r2'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0Controls the verbosity of information printed durin

In [68]:
gs.best_score_

np.float64(0.5921972919604244)